# NN-02 Backpropagation from Scratch

- Module: 06 Neural Networks
- Topic: chain rule on a small sigmoid network
- Estimated runtime: 6 minutes
- Prerequisites: the forward-pass notebook, binary cross-entropy, and the derivation in `derivations/backpropagation.md`
- Expected outputs: output-layer deltas, parameter gradients, and a successful numerical gradient check


## Learning Goals

By the end of this notebook, you should be able to:
- compute the output-layer error term for sigmoid plus binary cross-entropy;
- propagate that error backward to an earlier hidden layer; and
- validate the implementation with finite-difference gradient checking.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from notebook path.')


REPO_ROOT = find_repo_root(Path.cwd())
MODULE_SRC = REPO_ROOT / 'modules/06-neural-networks/src'
if str(MODULE_SRC) not in sys.path:
    sys.path.insert(0, str(MODULE_SRC))

from nn_from_scratch import ScratchMLP, train_test_split_indices

SEED = 7
rng = np.random.default_rng(SEED)
SEED


## Outline

1. Reuse a tiny network with explicit parameters.
2. Compute the forward pass and binary cross-entropy loss.
3. Match the backpropagation formulas from the derivation note.
4. Check the gradients numerically.


## Step 1 - Set up a small supervised batch

The output activation is sigmoid and the loss is binary cross-entropy, so the final-layer error simplifies to $\delta^{[2]} = A^{[2]} - Y$.


In [ ]:
X = np.array(
    [
        [0.20, -0.40],
        [1.10, 0.30],
        [-0.60, 0.50],
    ],
    dtype=float,
)
Y = np.array([[1.0], [0.0], [1.0]], dtype=float)

model = ScratchMLP(layer_dims=[2, 3, 1], activations=['tanh', 'sigmoid'], loss='binary_cross_entropy', seed=SEED)
model.parameters['W1'] = np.array([[0.80, -0.50], [0.30, 0.90], [-0.40, 0.70]], dtype=float)
model.parameters['b1'] = np.array([[0.10, -0.20, 0.05]], dtype=float)
model.parameters['W2'] = np.array([[1.20, -0.70, 0.40]], dtype=float)
model.parameters['b2'] = np.array([[0.15]], dtype=float)

Y_hat, caches = model.forward(X)
loss = model.compute_loss(Y_hat, Y)
{'Y_hat': np.round(Y_hat, 4), 'loss': round(loss, 6)}


## Step 2 - Write the layerwise error terms

For this batch shape convention:

- $A^{[1]} \in \mathbb{R}^{n \times d_1}$
- $W^{[2]} \in \mathbb{R}^{d_2 \times d_1}$
- $\delta^{[2]} \in \mathbb{R}^{n \times d_2}$

The batch gradients are

$$
\frac{\partial \mathcal{L}}{\partial W^{[2]}} = \frac{1}{n}(\delta^{[2]})^\top A^{[1]},
\qquad
\frac{\partial \mathcal{L}}{\partial b^{[2]}} = \frac{1}{n}\sum_{i=1}^n \delta^{[2]}_i.
$$


In [ ]:
A1 = caches[0]['A']
Z1 = caches[0]['Z']
W2 = model.parameters['W2']

# Delta2: (n, d2) with d2 = 1 for this binary classifier.
Delta2 = Y_hat - Y
# dW2: (d2, d1), db2: (1, d2)
dW2_manual = Delta2.T @ A1 / X.shape[0]
db2_manual = np.sum(Delta2, axis=0, keepdims=True) / X.shape[0]

{
    'Delta2': np.round(Delta2, 4),
    'dW2_manual': np.round(dW2_manual, 6),
    'db2_manual': np.round(db2_manual, 6),
}


## Step 3 - Push the error back through the hidden layer

The derivation note gives

$$
\delta^{[1]} = (\delta^{[2]} W^{[2]}) \odot (1 - \tanh^2(Z^{[1]})).
$$

Once $\delta^{[1]}$ is known, the first-layer parameter gradients follow the same pattern.


In [ ]:
# Delta2 @ W2 gives shape (n, d1); tanh derivative keeps the same shape.
Delta1 = (Delta2 @ W2) * (1.0 - np.square(np.tanh(Z1)))
# dW1: (d1, d0), db1: (1, d1)
dW1_manual = Delta1.T @ X / X.shape[0]
db1_manual = np.sum(Delta1, axis=0, keepdims=True) / X.shape[0]

manual_grads = {
    'dW1_manual': np.round(dW1_manual, 6),
    'db1_manual': np.round(db1_manual, 6),
    'dW2_manual': np.round(dW2_manual, 6),
    'db2_manual': np.round(db2_manual, 6),
}
manual_grads


## Step 4 - Compare against the library backward pass

If the implementation matches the derivation, the manual gradients and the reusable `backward` method should agree entrywise.


In [ ]:
backward_grads = model.backward(Y, caches)
comparison = {
    'dW1_match': bool(np.allclose(dW1_manual, backward_grads['dW1'])),
    'db1_match': bool(np.allclose(db1_manual, backward_grads['db1'])),
    'dW2_match': bool(np.allclose(dW2_manual, backward_grads['dW2'])),
    'db2_match': bool(np.allclose(db2_manual, backward_grads['db2'])),
}
comparison


## Step 5 - Validate with finite differences

Gradient checking perturbs one parameter at a time and estimates

$$
\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)}{2\varepsilon}.
$$

The relative errors should be very small if the algebra and the implementation are both correct.


In [ ]:
gradient_check_report = model.gradient_check(X, Y, epsilon=1e-6, max_checks_per_array=6)
gradient_check_report


## Pitfall

If the output layer uses sigmoid with binary cross-entropy, do not multiply by `sigmoid'(z)` again after already using the combined shortcut `A - Y`. That would double-count the derivative and produce gradients that are too small.


## Exercise

Replace the hidden activation with ReLU and adapt the hidden-layer derivative. Which entries of $\delta^{[1]}$ become exactly zero because the ReLU derivative vanishes there?


In [ ]:
relu_model = ScratchMLP(layer_dims=[2, 3, 1], activations=['relu', 'sigmoid'], loss='binary_cross_entropy', seed=SEED)
relu_model.parameters = {name: value.copy() for name, value in model.parameters.items()}
relu_pred, relu_caches = relu_model.forward(X)
relu_grads = relu_model.backward(Y, relu_caches)
{
    'relu_hidden_activation': np.round(relu_caches[0]['A'], 4),
    'relu_dW1': np.round(relu_grads['dW1'], 6),
}


## Summary

Backpropagation reused the forward-pass cache, matched the layerwise formulas from the derivation note, and passed a numerical gradient check. That is the core mechanical loop behind training.
